# 🖼️ 00 · 多模态大模型总览与 CLIP

> 多模态（图文/音视频）是 LLM 时代最火方向之一。本节先搭全局框架（对齐→理解→生成），
> 再**手写 CLIP 的对比学习目标 InfoNCE**——这是几乎所有多模态模型的基石。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 多模态包含哪些任务？ | 对齐 / 理解 / 生成 |
| 2 | CLIP 怎么训练？ | 图文对比学习（InfoNCE） |
| 3 | 对比学习损失怎么算？ | 相似度矩阵 + softmax |
| 4 | 训练好的模型能干嘛？ | zero-shot 分类 / 检索 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 多模态任务地图（必背框架）

In [2]:
tasks = [
    ('图文对齐', 'CLIP / ALIGN：把图文映射到同一向量空间，图文相似度可算'),
    ('视觉理解', 'BLIP / LLaVA / Qwen-VL：看图回答问题、图像对话'),
    ('视频理解', 'VideoLLM：视频抽帧→时序建模→问答/总结'),
    ('图文生成', 'SD / DiT / DALL·E：文本→图像；扩散模型为主'),
    ('统一模型', 'Gemini / GPT-4V / Qwen-VL：理解+生成+对话一体化'),
]
for i, (s, d) in enumerate(tasks, 1):
    print(f'  {i}. {s} —— {d}')
print('\n➡️ 面试串讲：先说任务分类，再说典型模型，最后讲训练范式')

  1. 图文对齐 —— CLIP / ALIGN：把图文映射到同一向量空间，图文相似度可算
  2. 视觉理解 —— BLIP / LLaVA / Qwen-VL：看图回答问题、图像对话
  3. 视频理解 —— VideoLLM：视频抽帧→时序建模→问答/总结
  4. 图文生成 —— SD / DiT / DALL·E：文本→图像；扩散模型为主
  5. 统一模型 —— Gemini / GPT-4V / Qwen-VL：理解+生成+对话一体化

➡️ 面试串讲：先说任务分类，再说典型模型，最后讲训练范式


## 2️⃣ 构造"图文对"特征（模拟 embedding）

In [3]:
# 模拟 64 个图文对：图像 embedding 与文本 embedding（各 32 维）
n_pairs, dim = 64, 32
# 确定性共享向量 + 独立噪声：配对相似、非配对随机
shared = rng.normal(0, 1, (n_pairs, dim))
img_emb = shared + rng.normal(0, 0.6, (n_pairs, dim))
txt_emb = shared + rng.normal(0, 0.6, (n_pairs, dim))
img_emb /= np.linalg.norm(img_emb, axis=1, keepdims=True)
txt_emb /= np.linalg.norm(txt_emb, axis=1, keepdims=True)
print('图像 embedding:', img_emb.shape, '文本 embedding:', txt_emb.shape)
print('➡️ 真实 CLIP 中：图像来自 ViT 编码器，文本来自 Transformer 编码器')

图像 embedding: (64, 32) 文本 embedding: (64, 32)
➡️ 真实 CLIP 中：图像来自 ViT 编码器，文本来自 Transformer 编码器


## 3️⃣ 相似度矩阵与 InfoNCE 损失（手写）

In [4]:
def infonce_loss(img_emb, txt_emb, tau=0.07):
    # 归一化后点积 = 余弦相似度
    sim = img_emb @ txt_emb.T / tau          # (N, N)
    # 图像→文本：每行对角线为正样本，其余为负样本
    logits_i2t = sim - np.max(sim, axis=1, keepdims=True)  # 数值稳定
    exp_i2t = np.exp(logits_i2t)
    prob_i2t = exp_i2t / exp_i2t.sum(axis=1, keepdims=True)
    loss_i2t = -np.mean(np.log(prob_i2t[np.arange(len(img_emb)), np.arange(len(img_emb))] + 1e-9))
    # 对称：文本→图像
    loss_t2i = loss_i2t  # 这里用同一矩阵（对称演示）
    return loss_i2t, sim / tau

loss, sim = infonce_loss(img_emb, txt_emb)
print(f'InfoNCE 损失 = {loss:.4f}')
print('相似度矩阵形状:', sim.shape)
# 检查对角线是否确实是最大值（配对相似）
diag_max_ratio = (sim.argmax(axis=1) == np.arange(len(img_emb))).mean()
print(f'配对是行内最大相似度的比例: {diag_max_ratio:.1%}')
assert diag_max_ratio > 0.9
print('✅ 对比学习目标正确提取了配对关系')

InfoNCE 损失 = 0.0741
相似度矩阵形状: (64, 64)
配对是行内最大相似度的比例: 100.0%
✅ 对比学习目标正确提取了配对关系


## 4️⃣ 温度参数 τ 的作用

In [5]:
print('''温度 τ 的作用：
  τ 大（如 1.0）→ 相似度分布平坦 → 梯度小、训练"温柔"；
  τ 小（如 0.07）→ 分布尖锐 → 突出正负样本差异、训练更"激进"；
  CLIP 用 τ≈0.07；τ 过小会不稳定（梯度爆炸）。
''')
# 演示：不同温度下的 softmax 分布
import scipy.special as sp
s = np.array([0.9, 0.8, 0.3, -0.1])   # 正样本 0.9，其余为负
for tau in [1.0, 0.5, 0.07]:
    p = sp.softmax(s / tau)
    print(f'τ={tau:<5} -> softmax = {np.round(p, 3)} 正样本概率={p[0]:.3f}')

温度 τ 的作用：
  τ 大（如 1.0）→ 相似度分布平坦 → 梯度小、训练"温柔"；
  τ 小（如 0.07）→ 分布尖锐 → 突出正负样本差异、训练更"激进"；
  CLIP 用 τ≈0.07；τ 过小会不稳定（梯度爆炸）。



τ=1.0   -> softmax = [0.354 0.321 0.195 0.13 ] 正样本概率=0.354
τ=0.5   -> softmax = [0.443 0.363 0.134 0.06 ] 正样本概率=0.443
τ=0.07  -> softmax = [0.807 0.193 0.    0.   ] 正样本概率=0.807


## 5️⃣ Zero-shot 分类：CLIP 怎么"零样本"干活

In [6]:
# 思路：把类别名变成文本模板，图像与所有类别文本比相似度，取最高
classes = ['一只猫', '一只狗', '一辆汽车', '一棵树']
n_cls = len(classes)
# 模拟：图像与各类别文本的相似度（类 0 "猫" 最高）
sim_cls = np.array([1.1, 0.2, -0.3, 0.1])
probs = sp.softmax(sim_cls / 0.07)
for c, p in zip(classes, probs):
    print(f'{c}: {p:.1%}')
print(f'预测类别: {classes[np.argmax(probs)]}')
assert np.argmax(probs) == 0
print('\n✅ zero-shot 分类 = 文本模板 + 相似度比较，无需训练')

一只猫: 100.0%
一只狗: 0.0%
一辆汽车: 0.0%
一棵树: 0.0%
预测类别: 一只猫

✅ zero-shot 分类 = 文本模板 + 相似度比较，无需训练


## 6️⃣ CLIP 的局限（面试高频）

In [7]:
print('''1) 数据要求高：需海量图文对（CLIP 4 亿对），清洗成本大；
2) 网格级能力弱：对细粒度属性（颜色/数量/位置）理解差；
3) zero-shot 偏置：对常见类别好、长尾差（ImageNet 强、细粒度弱）；
4) 生成能力缺失：CLIP 只做表示/对比，不生成图像；
5) 中文支持一般：预训练数据英文为主，中文本地化需二次训练。
''')

1) 数据要求高：需海量图文对（CLIP 4 亿对），清洗成本大；
2) 网格级能力弱：对细粒度属性（颜色/数量/位置）理解差；
3) zero-shot 偏置：对常见类别好、长尾差（ImageNet 强、细粒度弱）；
4) 生成能力缺失：CLIP 只做表示/对比，不生成图像；
5) 中文支持一般：预训练数据英文为主，中文本地化需二次训练。



## 📝 自测清单

- [ ] 能画出多模态任务地图（对齐/理解/生成/统一）
- [ ] 能默写 InfoNCE 损失公式并解释温度 τ
- [ ] 能解释 CLIP 为什么能 zero-shot 分类
- [ ] 能说出 CLIP 五大局限
- [ ] 能回答"多模态与单模态各自难点在哪"（对齐难题）